# Visual Wake Word - FP4 Model Compression & Evaluation

This notebook re-runs MX-FP4 compression (with fixes) and evaluates all VWW models on COCO minival.

**Before running:**
1. Make sure your Google Drive has `val2014/` and `annotations/` folders (with `instances_val2014.json`)
2. Optionally provide `train2014/` for proper TFLite calibration
3. Update the paths in Cell 2 if your folders are in a different location

In [ ]:
#@title 1. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools opencv-python-headless
print("Done.")

In [ ]:
#@title 2. Configure Paths { display-mode: "form" }
import os

#@markdown **Google Drive paths to your COCO data:**
val2014_folder = "/content/drive/MyDrive/val2014" #@param {type:"string"}
annotations_folder = "/content/drive/MyDrive/annotations" #@param {type:"string"}
train2014_folder = "/content/drive/MyDrive/train2014" #@param {type:"string"}

# Verify val paths exist
assert os.path.isdir(val2014_folder), f"val2014 folder not found: {val2014_folder}"
assert os.path.isdir(annotations_folder), f"annotations folder not found: {annotations_folder}"

ann_file = os.path.join(annotations_folder, 'instances_val2014.json')
assert os.path.isfile(ann_file), f"instances_val2014.json not found in {annotations_folder}"

num_images = len([f for f in os.listdir(val2014_folder) if f.endswith('.jpg')])
print(f"Found {num_images} val images in {val2014_folder}")
print(f"Found annotations at {ann_file}")

# Check train2014 (optional, for TFLite calibration)
has_train = os.path.isdir(train2014_folder)
if has_train:
    num_train = len([f for f in os.listdir(train2014_folder) if f.endswith('.jpg')])
    print(f"Found {num_train} training images in {train2014_folder}")
else:
    print(f"train2014 not found at {train2014_folder} — will use synthetic calibration data")

print("All good!")

In [ ]:
#@title 3. Create symlinked data directory & clone repo
import os

# Create the directory structure eval_fp4_coco.py expects
os.makedirs('/content/coco/raw-data', exist_ok=True)

# Symlink to Drive folders (no copying needed!)
for name, src in [('val2014', val2014_folder), ('annotations', annotations_folder)]:
    dst = f'/content/coco/raw-data/{name}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Symlink train2014 if available
if has_train and not os.path.exists('/content/coco/raw-data/train2014'):
    os.symlink(train2014_folder, '/content/coco/raw-data/train2014')

print("Symlinks created:")
!ls -la /content/coco/raw-data/

# Clone repo (use the feature branch with evaluation scripts)
BRANCH = "claude/explain-codebase-mljz2na9k1pqxa7f-C4z0d"
!git clone -b {BRANCH} https://github.com/amitmate/visualwakeword.git /content/vww 2>/dev/null || (cd /content/vww && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH})

# Copy minival IDs
import shutil
shutil.copy('/content/vww/mscocominival.txt', '/content/coco/mscocominival.txt')
print(f"\nRepo cloned (branch: {BRANCH}). Ready!")

In [ ]:
#@title 4. Re-run FP4 compression (with fixes: kernels-only + real calibration data)
%cd /content/vww

train_arg = f"--train-dir /content/coco/raw-data/train2014" if has_train else ""
!python compress_fp4.py {train_arg}

In [ ]:
#@title 5. Run Evaluation on COCO minival
%cd /content/vww
!python eval_fp4_coco.py \
    --data-dir /content/coco/raw-data \
    --minival-ids /content/coco/mscocominival.txt \
    --h5-model modelVisualWakeWord.h5 \
    --tflite-model modelVisualWakeWord.tflite \
    --fp4-bin compressed_models/model_mxfp4.fp4bin \
    --fp4-tflite compressed_models/model_mxfp4_int8.tflite